# Moderación semiautomática de videos peruanos de YouTube mediante modelos clásicos y neuronales de procesamiento del lenguaje natural

**Trabajo final del curso de Procesamiento de Lenguaje Natural (PLN) de la Maestría en Inteligencia Artificial de la Universidad Nacional de Ingeniería (UNI) — Semestre 2026-1**

**Grupo 4:** Luis Enrique Koc Góngora, Alex Felipe Mancilla Antay, Herbert Antonio Meléndez García y Dennis Jack Paitán Cano

---

## 03 · Transformer jerárquico multitarea

Entrena o audita el contrato de etiquetas v2.1 sin consultar test para seleccionar modelos, épocas o umbrales.

El aprendizaje multitarea comparte representaciones entre objetivos relacionados [1], mientras la clasificación jerárquica [2] y multietiqueta [3] aportan marcos generales para salidas dependientes o simultáneas. Las cinco salidas principales, las auxiliares finas, los flags y la ponderación de sus pérdidas son un diseño local y no heredan garantías de transferencia positiva.

**Contrato de etiquetas v2.1:** cinco salidas entrenadas: `SEGURO`, `RACISMO_DISCRIMINACION`, `ATAQUE_POR_GENERO_IDENTIDAD`, `ACOSO_AMENAZA` y `CONTENIDO_SEXUAL`. `SEGURO` es excluyente; las cuatro categorías de daño son multietiqueta y pueden coexistir. Los casos indeterminados se difieren y no entran al entrenamiento. Esta combinación, sus umbrales y sus reglas de exclusividad son decisiones operativas locales.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. En local no instala paquetes. En Colab, únicamente la celda de bootstrap instala versiones fijadas desde el bundle SHA-256 de Drive. No usa rutas personales. Revise el README de esta etapa.

## Backend opcional Google Colab L4 desde VS Code

Instale la extensión oficial **Google Colab** (`google.colab`), seleccione `Select Kernel > Colab` y asigne una **NVIDIA L4**. El notebook permanece local; Drive transporta solo el bundle mínimo verificado. Edite `COLAB_RUN_ID` para separar experimentos. La compatibilidad de `drive.mount()` desde VS Code requiere la extensión v0.2.1 o posterior [4]. La integridad del bundle se comprueba con SHA-256 [5]. No sincronice cachés de modelos ni escriba checkpoints directamente en Drive.

In [ ]:
# Backend reproducible: local o Google Colab desde VS Code
from pathlib import Path
import hashlib
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import zipfile

COLAB_NOTEBOOK_ID = "03_04"
COLAB_DRIVE_FOLDER = "ModeracionPeru_Colab"  # Debe coincidir con config/colab_l4.json
COLAB_RUN_ID = ""  # Vacío reanuda <notebook>_working_v2_1; use otro ID para otro experimento
COLAB_REQUIRE_L4 = True
IN_COLAB = importlib.util.find_spec("google.colab") is not None
COLAB_CONTEXT = None

def _sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while block := handle.read(1024 * 1024):
            digest.update(block)
    return digest.hexdigest()

def _find_local_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("No se encontró pyproject.toml")

if IN_COLAB:
    from google.colab import drive

    # La extensión oficial de Colab para VS Code admite drive.mount desde v0.2.1.
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive") / COLAB_DRIVE_FOLDER
    BUNDLE_DIR = DRIVE_ROOT / "bundle"
    manifest_path = BUNDLE_DIR / "bundle_manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(
            "Falta bundle_manifest.json. Prepare y suba resultados/colab_bundle a " + str(BUNDLE_DIR)
        )
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    core = BUNDLE_DIR / manifest["core"]["name"]
    if _sha256(core) != manifest["core"]["sha256"]:
        raise ValueError("project_core.zip no coincide con el manifiesto SHA-256")

    RUNTIME_ROOT = Path("/content/moderacion_peru")
    ROOT = RUNTIME_ROOT / "project"
    marker = RUNTIME_ROOT / ".core_sha256"
    expected_core = manifest["core"]["sha256"]
    if not ROOT.is_dir() or not marker.is_file() or marker.read_text().strip() != expected_core:
        if ROOT.exists():
            shutil.rmtree(ROOT)
        ROOT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(core) as archive:
            archive.extractall(ROOT)
        os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements/colab-l4.txt")]
        )
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(ROOT)])
        marker.parent.mkdir(parents=True, exist_ok=True)
        marker.write_text(expected_core + "\n", encoding="utf-8")

    os.environ["MODPERU_ROOT"] = str(ROOT)
    os.environ["HF_HOME"] = "/content/huggingface"
    importlib.invalidate_caches()
    if str(ROOT / "src") not in sys.path:
        sys.path.insert(0, str(ROOT / "src"))
    from moderacion_peru.colab import colab_runtime_diagnostics, prepare_colab_context

    COLAB_CONTEXT = prepare_colab_context(
        COLAB_NOTEBOOK_ID,
        project_root=ROOT,
        drive_root=DRIVE_ROOT,
        runtime_root=RUNTIME_ROOT,
        run_id=COLAB_RUN_ID or None,
        require_l4=COLAB_REQUIRE_L4,
        resume=True,
    )
    from moderacion_peru.notebook_ui import show_callout, show_command, show_result, show_summary, show_table
    show_result('Diagnóstico de Colab', colab_runtime_diagnostics(), tone='success')
    show_result('Contexto reproducible', COLAB_CONTEXT.as_dict(), tone='success')
else:
    ROOT = _find_local_root()
    if str(ROOT / "src") not in sys.path:
        sys.path.insert(0, str(ROOT / "src"))
    from moderacion_peru.notebook_ui import show_callout, show_command, show_result, show_summary, show_table
    show_summary('Entorno del proyecto', {'raíz': ROOT, 'backend': 'local'}, tone='success')


## Restauración reproducible del dataset

In [ ]:
from moderacion_peru.colab import prepare_local_bundle_input

if globals().get('COLAB_CONTEXT') is None:
    dataset_checkpoint = prepare_local_bundle_input('dataset_5_salidas', project_root=ROOT)
else:
    dataset_path = COLAB_CONTEXT.input('dataset_5_salidas')
    dataset_checkpoint = {
        'status': 'verified_in_colab',
        'input_key': 'dataset_5_salidas',
        'path': dataset_path,
        'bytes': dataset_path.stat().st_size,
    }
show_result('Dataset descomprimido y verificado', dataset_checkpoint, tone='success')


## Configuración y ejecución

In [ ]:
from moderacion_peru.experiments import train_neural_experiment
DATA=COLAB_CONTEXT.input('dataset_5_salidas') if COLAB_CONTEXT else ROOT/'datos/model_ready/v2/dataset_5_salidas.jsonl'
OUTPUT_ROOT=COLAB_CONTEXT.scratch_output_dir if COLAB_CONTEXT else ROOT/'modelos/v2/multitarea'
DEVICE='cuda' if COLAB_CONTEXT else 'auto'
RUN_TRAINING=False
if RUN_TRAINING:
    show_result('Entrenamiento multitarea', train_neural_experiment(DATA,OUTPUT_ROOT,experiment='multitask',device=DEVICE), tone='success')
else:
    show_summary('Configuración preliminar', {'datos': DATA, 'salida': OUTPUT_ROOT, 'dispositivo': DEVICE, 'salidas_principales': 5, 'auxiliares': '14 finas + 3 flags', 'acción': 'RUN_TRAINING=True completa el ciclo o devuelve noop.'}, tone='neutral')

## Publicación o checkpoint en Drive

Los archivos se generan en el SSD efímero de `/content`. Active esta celda después de un checkpoint coherente o al finalizar; publica un solo TAR.GZ y luego su manifiesto.

In [ ]:
PUBLISH_TO_DRIVE = False
if COLAB_CONTEXT is not None and PUBLISH_TO_DRIVE:
    from moderacion_peru.colab import publish_colab_outputs
    show_result('Publicación en Drive', publish_colab_outputs(COLAB_CONTEXT), tone='success')
elif COLAB_CONTEXT is not None:
    show_callout('Publicación desactivada', 'Cambie PUBLISH_TO_DRIVE=True tras guardar un checkpoint consistente.', tone='neutral')
else:
    show_callout('Backend local', 'Los artefactos ya permanecen en el workspace.', tone='success')

## Referencias

[1] R. Caruana, "Multitask Learning," Mach. Learn., vol. 28, pp. 41–75, 1997, doi: 10.1023/A:1007379606734.

[2] C. N. Silla Jr. and A. A. Freitas, "A Survey of Hierarchical Classification Across Different Application Domains," Data Min. Knowl. Discovery, vol. 22, no. 1–2, pp. 31–72, 2011, doi: 10.1007/s10618-010-0175-9.

[3] G. Tsoumakas and I. Katakis, "Multi-Label Classification: An Overview," Int. J. Data Warehousing Mining, vol. 3, no. 3, pp. 1–13, 2007, doi: 10.4018/jdwm.2007070101.

[4] Google Colab, "Known Issues and Workarounds," googlecolab/colab-vscode Wiki, 2026. [Online]. Available: https://github.com/googlecolab/colab-vscode/wiki/Known-Issues-and-Workarounds. Accessed: Aug. 5, 2026.

[5] National Institute of Standards and Technology, "Secure Hash Standard (SHS)," FIPS PUB 180-4, Aug. 2015, doi: 10.6028/NIST.FIPS.180-4.